# 03 — Arricchimento Testi EUR-Lex: Testo Integrale + Segmentazione

Recupera il **testo integrale** di ogni atto normativo del grafo focale da EUR-Lex
e produce una **segmentazione strutturata** per articolo e considerando.

## Cache globale cross-materia

I testi estratti vengono accumulati in **`data/processed/nodes_texts.csv`**.
Ad ogni esecuzione — anche su una materia diversa — il notebook controlla prima
questa cache: se un CELEX è già stato estratto in precedenza, il suo testo viene
riutilizzato senza toccare EUR-Lex.

```
data/processed/nodes_texts.csv          ← cache globale (tutti i CELEX mai estratti)
data/output/{materia}/nodes_texts.csv   ← sottoinsieme per questa materia
data/output/{materia}/texts_checkpoint.csv  ← checkpoint fetch in corso
```

## Cosa viene estratto

| Sezione | Contenuto | Segmentazione |
|---|---|---|
| **Titolo** | Testo completo del titolo | — (metadato) |
| **Preambolo** | Tutti i considerando | Un segmento per considerando |
| **Articoli** | Testo integrale di **tutti** gli articoli | Un segmento per articolo |
| **Allegati** | Testo **integrale** di ogni allegato | Un segmento per allegato |

## 0. Configurazione

**Modifica solo questa cella.** Il resto del notebook gira in automatico.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
#  MATERIA  →  nome della cartella in data/output/  (output del notebook 02)
# ─────────────────────────────────────────────────────────────────────────────

MATERIA_NAME = "fdi_screening"   # <- stessa cartella scelta nel notebook 02


# ── Parametri fetch ───────────────────────────────────────────────────────────
DELAY_SECONDS    = 0.7    # pausa tra richieste (rispetta il rate limit EUR-Lex)
CHECKPOINT_EVERY = 50     # salva checkpoint ogni N nodi
TIMEOUT          = 20     # secondi per ciascuna richiesta HTTP


# ── Parametri segmentazione ───────────────────────────────────────────────────
MAX_TITLE_CHARS     = 500   # limite solo per il titolo (metadato, non entra nel clustering)
SEGMENT_SPLIT_CHARS = 800   # articoli più lunghi di questo vengono spezzati per paragrafo

## 1. Setup Percorsi e Costanti

In [2]:
import pandas as pd
import time
import os
import re
import json
from bs4 import BeautifulSoup
import eurlex

# ── Percorsi ─────────────────────────────────────────────────────────────────
output_path       = os.path.join('..', 'data', 'output', MATERIA_NAME)
input_file        = os.path.join(output_path, 'nodes_focal.csv')
output_file       = os.path.join(output_path, 'nodes_texts.csv')
ckpt_file         = os.path.join(output_path, 'texts_checkpoint.csv')
# Cache globale — percorso fisso, condiviso tra tutte le materie
global_cache_file = os.path.join('..', 'data', 'processed', 'nodes_texts.csv')

os.makedirs(os.path.join('..', 'data', 'processed'), exist_ok=True)

# ── Marker fine preambolo ─────────────────────────────────────────────────────
PREAMBLE_END_MARKERS = [
    'HAVE ADOPTED THIS REGULATION:',
    'HAS ADOPTED THIS REGULATION:',
    'HAVE ADOPTED THIS DIRECTIVE:',
    'HAS ADOPTED THIS DIRECTIVE:',
    'HAVE ADOPTED THIS DECISION:',
    'HAS ADOPTED THIS DECISION:',
    'HAVE ADOPTED THIS FRAMEWORK DECISION:',
    'HAVE ADOPTED THIS RECOMMENDATION:',
    'HEREBY DECIDES:',
    'HAS DECIDED AS FOLLOWS:',
    'HEREBY RECOMMENDS:',
    'IS OF THE OPINION THAT:',
    'HAVE AGREED AS FOLLOWS:',
    'HAVE DECIDED AS FOLLOWS:',
]

# ── Colonne testo ─────────────────────────────────────────────────────────────
TEXT_COLS = [
    'title', 'preamble', 'articles', 'annexes',
    'full_text', 'segments', 'n_segments',
    'sections_found', 'text_status', 'text_length',
]

print(f"Materia:       {MATERIA_NAME}")
print(f"Input:         {input_file}")
print(f"Output:        {output_file}")
print(f"Checkpoint:    {ckpt_file}")
print(f"Cache globale: {global_cache_file}")

Materia:       fdi_screening
Input:         ..\data\output\fdi_screening\nodes_focal.csv
Output:        ..\data\output\fdi_screening\nodes_texts.csv
Checkpoint:    ..\data\output\fdi_screening\texts_checkpoint.csv
Cache globale: ..\data\processed\nodes_texts.csv


## 2. Caricamento Nodi, Cache Globale e Checkpoint

**Ordine di priorità** per decidere se un CELEX va estratto o no:

1. **Cache globale** (`data/processed/nodes_texts.csv`) — CELEX già estratti in qualsiasi materia precedente → riutilizzati direttamente
2. **Checkpoint locale** (`data/output/{materia}/texts_checkpoint.csv`) — estratti in questa sessione ma non ancora in cache
3. **Da scrapare** — tutto il resto

> Se esiste un checkpoint della versione precedente del notebook (colonna `full_text_excerpt` invece di `full_text`), eliminarlo prima di procedere.

In [3]:
nodes     = pd.read_csv(input_file)
celex_col = 'Label' if 'Label' in nodes.columns else 'celex_id'
print(f"Nodi nel grafo focale: {len(nodes)}")

# ── Cache globale ─────────────────────────────────────────────────────────────
if os.path.exists(global_cache_file):
    global_cache = pd.read_csv(global_cache_file, low_memory=False)
    if 'full_text' not in global_cache.columns and 'full_text_excerpt' in global_cache.columns:
        print("\u26a0\ufe0f  Cache globale con formato vecchio (full_text_excerpt). Ignorata.")
        global_cache   = pd.DataFrame(columns=['Label'] + TEXT_COLS)
    cached_celexes = set(global_cache['Label'].dropna().astype(str))
    print(f"Cache globale:  {len(global_cache)} CELEX gi\u00e0 estratti")
else:
    global_cache   = pd.DataFrame(columns=['Label'] + TEXT_COLS)
    cached_celexes = set()
    print("Cache globale:  non trovata, verr\u00e0 creata")

# ── Checkpoint locale ─────────────────────────────────────────────────────────
if os.path.exists(ckpt_file):
    checkpoint = pd.read_csv(ckpt_file, low_memory=False)
    if 'full_text' not in checkpoint.columns:
        print("\u26a0\ufe0f  Checkpoint locale con formato vecchio. Eliminarlo e rieseguire.")
        raise RuntimeError(f"Checkpoint incompatibile: {ckpt_file}")
    ckpt_celexes = set(checkpoint['Label'].dropna().astype(str))
    print(f"Checkpoint:     {len(checkpoint)} nodi gi\u00e0 processati")
else:
    checkpoint   = pd.DataFrame(columns=['Id', 'Label'] + TEXT_COLS)
    ckpt_celexes = set()
    print("Checkpoint:     non trovato, si parte da zero")

# ── Nodi da scrapare ──────────────────────────────────────────────────────────
all_celexes      = set(nodes[celex_col].dropna().astype(str))
already_covered  = cached_celexes | ckpt_celexes
celexes_to_fetch = all_celexes - already_covered
nodes_todo       = nodes[nodes[celex_col].astype(str).isin(celexes_to_fetch)].copy()

print(f"\nRiepilogo:")
print(f"  Da cache globale:  {len(all_celexes & cached_celexes)}")
print(f"  Da checkpoint:     {len(all_celexes & ckpt_celexes - cached_celexes)}")
print(f"  Da scrapare ora:   {len(nodes_todo)}")

Nodi nel grafo focale: 19
Cache globale:  1794 CELEX già estratti
Checkpoint:     non trovato, si parte da zero

Riepilogo:
  Da cache globale:  19
  Da checkpoint:     0
  Da scrapare ora:   0


## 3. Fetch HTML da EUR-Lex

In [4]:
def fetch_eurlex_html(celex, timeout=TIMEOUT):
    if pd.isna(celex) or str(celex).strip() == '':
        return None, 'not_found'
    celex = str(celex).strip()
    try:
        html = eurlex.get_html_by_celex_id(celex, language='en')
        if not html or len(html) < 500:
            return None, 'not_found'
        STRUCTURAL_TAGS = ['eli-subdivision', 'oj-doc-ti', 'doc-ti']
        if not any(tag in html for tag in STRUCTURAL_TAGS):
            return None, 'no_structure'
        return html, 'ok'
    except Exception as e:
        err = str(e).lower()
        if 'timeout' in err:                   return None, 'timeout'
        if '404' in err or 'not found' in err: return None, 'not_found'
        return None, 'error'


print("Test fetch su 3 atti...")
for celex_test in ['32019R0452', '32008L0114', '11957E']:
    if celex_test in cached_celexes:
        print(f"  {celex_test:<20} \u2192 (gi\u00e0 in cache globale, skip)")
        continue
    html_test, status_test = fetch_eurlex_html(celex_test)
    size = f"{len(html_test):,} car." if html_test else '\u2014'
    print(f"  {celex_test:<20} \u2192 {status_test:<20} {size}")
    time.sleep(1)

Test fetch su 3 atti...
  32019R0452           → (già in cache globale, skip)
  32008L0114           → (già in cache globale, skip)
  11957E               → no_structure         —


## 4. Funzioni di Estrazione Testo Integrale

In [5]:
def extract_title(soup):
    for css_class in ['oj-doc-ti', 'doc-ti']:
        tags = soup.find_all('p', class_=css_class)
        if not tags:
            continue
        parti = []
        for tag in tags:
            testo = tag.get_text(separator=' ', strip=True)
            if testo.startswith(('ANNEX', 'SCHEDULE', 'APPENDIX', 'ALLEGATO')):
                break
            parti.append(testo)
        if parti:
            return ' '.join(parti)[:MAX_TITLE_CHARS]
    title_tag = soup.find('title')
    if title_tag:
        text = re.sub(r'\s*[-\u2013|]\s*EUR-Lex.*$', '',
                      title_tag.get_text(strip=True), flags=re.IGNORECASE)
        if len(text) > 20:
            return text[:MAX_TITLE_CHARS]
    return None


def extract_preamble_segments(soup):
    subdivisions = soup.find_all(class_='eli-subdivision')
    if not subdivisions:
        return []
    testo = re.sub(r'\s+', ' ',
                   subdivisions[0].get_text(separator=' ', strip=True)).strip()
    for marker in PREAMBLE_END_MARKERS:
        idx = testo.find(marker)
        if idx != -1:
            testo = testo[:idx].strip()
            break
    if len(testo) < 50:
        return []
    parts    = re.split(r'\((\d+)\)\s+', testo)
    segments = []
    if len(parts) <= 1:
        segments.append({'tipo': 'preambolo', 'identificatore': '0', 'testo': testo})
        return segments
    header = parts[0].strip()
    if header and len(header) >= 30:
        segments.append({'tipo': 'preambolo_header', 'identificatore': '0', 'testo': header})
    i = 1
    while i < len(parts) - 1:
        num, corpo = parts[i].strip(), parts[i + 1].strip()
        if corpo and len(corpo) >= 20:
            segments.append({'tipo': 'considerando', 'identificatore': num, 'testo': corpo})
        i += 2
    return segments


def extract_articles_full(soup):
    article_divs = []
    for div in soup.find_all(class_='eli-subdivision'):
        if re.match(r'^art_\d+', div.get('id', '')):
            article_divs.append(div)
    if not article_divs:
        article_divs = soup.find_all(class_='eli-subdivision',
                                     attrs={'data-section': 'article'})
    if not article_divs:
        for div in soup.find_all(class_='eli-subdivision')[1:]:
            if re.match(r'^Article\s+\d+',
                        div.get_text(separator=' ', strip=True)[:100], re.IGNORECASE):
                article_divs.append(div)
    segments = []
    for div in article_divs:
        testo = re.sub(r'\s+', ' ', div.get_text(separator=' ', strip=True)).strip()
        if len(testo) < 10:
            continue
        m   = re.match(r'Article\s+(\d+[a-z]?)', testo, re.IGNORECASE)
        num = m.group(1) if m else re.sub(r'^art_', '', div.get('id', str(len(segments)+1)))
        segments.append({'tipo': 'articolo', 'identificatore': num, 'testo': testo})
    return segments


def extract_annexes_full(soup):
    annex_patterns = [r'^anx_', r'^ann_', r'^annex']
    annex_divs = [
        div for div in soup.find_all(class_='eli-subdivision')
        if any(re.match(p, div.get('id', '').lower()) for p in annex_patterns)
    ]
    if not annex_divs:
        for div in soup.find_all(class_='eli-subdivision'):
            if re.match(r'^(ANNEX|APPENDIX|SCHEDULE)\b',
                        div.get_text(separator=' ', strip=True)[:60], re.IGNORECASE):
                annex_divs.append(div)
    segments = []
    for div in annex_divs:
        testo = re.sub(r'\s+', ' ', div.get_text(separator=' ', strip=True)).strip()
        if len(testo) < 10:
            continue
        m     = re.match(r'(?:ANNEX|APPENDIX|SCHEDULE)\s+([IVXivx\d]+[A-Za-z]?)',
                         testo[:80], re.IGNORECASE)
        ident = m.group(1).upper() if m else str(len(segments)+1)
        segments.append({'tipo': 'allegato', 'identificatore': ident, 'testo': testo})
    return segments


def split_article_by_paragraph(seg, max_chars=SEGMENT_SPLIT_CHARS):
    testo = seg['testo']
    if len(testo) <= max_chars:
        return [seg]
    paragraphs = [p.strip() for p in re.split(
        r'(?=(?:^|\s)(?:\d+\.\s|\([a-z]\)\s|\([ivx]+\)\s))', testo)
        if p.strip() and len(p.strip()) > 30]
    if len(paragraphs) <= 1:
        return [seg]
    return [
        {'tipo': seg['tipo'],
         'identificatore': f"{seg['identificatore']}_p{i}",
         'testo': para}
        for i, para in enumerate(paragraphs, start=1)
    ]


def build_segments(preamble_segs, article_segs, annex_segs):
    all_segs = list(preamble_segs)
    for seg in article_segs:
        all_segs.extend(split_article_by_paragraph(seg))
    all_segs.extend(annex_segs)
    for i, seg in enumerate(all_segs):
        seg['segment_id'] = i
    return all_segs


print("Funzioni di estrazione definite.")

Funzioni di estrazione definite.


## 5. Funzione Principale

In [6]:
def extract_all_sections(celex):
    empty = {'title': None, 'preamble': None, 'articles': None, 'annexes': None,
             'full_text': None, 'segments': None, 'n_segments': 0,
             'sections_found': '', 'text_status': None, 'text_length': 0}
    html, fetch_status = fetch_eurlex_html(celex)
    if fetch_status != 'ok':
        return {**empty, 'text_status': fetch_status}
    try:
        soup = BeautifulSoup(html, 'html.parser')
    except Exception:
        return {**empty, 'text_status': 'parse_error'}

    title         = extract_title(soup)
    preamble_segs = extract_preamble_segments(soup)
    article_segs  = extract_articles_full(soup)
    annex_segs    = extract_annexes_full(soup)

    preamble_text = ' \n '.join(s['testo']   for s in preamble_segs) if preamble_segs else None
    articles_text = ' \n\n '.join(s['testo'] for s in article_segs)  if article_segs  else None
    annexes_text  = ' \n\n '.join(s['testo'] for s in annex_segs)    if annex_segs    else None

    segments = build_segments(preamble_segs, article_segs, annex_segs)
    sections_found = ','.join(filter(None, [
        'title' if title else None, 'preamble' if preamble_segs else None,
        'articles' if article_segs else None, 'annexes' if annex_segs else None,
    ]))
    parti = []
    if title:         parti.append(f"[TITLE] {title}")
    if preamble_text: parti.append(f"[PREAMBLE] {preamble_text}")
    if articles_text: parti.append(f"[ARTICLES] {articles_text}")
    if annexes_text:  parti.append(f"[ANNEXES] {annexes_text}")
    if not parti:
        return {**empty, 'text_status': 'no_content'}

    full_text = ' \n\n '.join(parti)
    return {
        'title': title, 'preamble': preamble_text,
        'articles': articles_text, 'annexes': annexes_text,
        'full_text': full_text,
        'segments': json.dumps(segments, ensure_ascii=False),
        'n_segments': len(segments),
        'sections_found': sections_found,
        'text_status': 'ok',
        'text_length': len(full_text),
    }


print("Funzione principale definita.")

Funzione principale definita.


## 6. Test su Campione

In [7]:
from collections import Counter

for celex in ['32019R0452', '32008L0114', '62019CJ0079']:
    print(f"{'='*60}")
    if celex in cached_celexes:
        result = global_cache[global_cache['Label'] == celex].iloc[0][TEXT_COLS].to_dict()
        source = '(da cache globale)'
    else:
        result = extract_all_sections(celex)
        source = '(da EUR-Lex)'
        time.sleep(1)
    print(f"CELEX: {celex}  {source}")
    print(f"  Status: {result.get('text_status')} | "
          f"Testo: {result.get('text_length', 0):,} car. | "
          f"Segmenti: {result.get('n_segments', 0)}")
    segs_raw = result.get('segments')
    if segs_raw and pd.notna(segs_raw):
        segs = json.loads(segs_raw)
        print(f"  Distribuzione: {dict(Counter(s['tipo'] for s in segs))}")
    print()

CELEX: 32019R0452  (da cache globale)
  Status: ok | Testo: 46,637 car. | Segmenti: 142
  Distribuzione: {'preambolo_header': 1, 'considerando': 47, 'articolo': 94}

CELEX: 32008L0114  (da cache globale)
  Status: ok | Testo: 24,757 car. | Segmenti: 66
  Distribuzione: {'preambolo_header': 1, 'considerando': 21, 'articolo': 44}

CELEX: 62019CJ0079  (da EUR-Lex)
  Status: no_structure | Testo: 0 car. | Segmenti: 0



## 7. Fetch Completo con Checkpoint e Aggiornamento Cache

Ogni batch viene salvato sia nel **checkpoint locale** che nella **cache globale**.
Se tutti i nodi sono già coperti, la cella termina immediatamente.

In [8]:
def _save_checkpoint_and_cache(new_results_df):
    """
    Aggiorna checkpoint locale e cache globale con i nuovi risultati.
    La cache globale e' append-only: i CELEX gia' presenti non vengono sovrascritti.
    """
    global checkpoint, global_cache

    # Checkpoint locale (per questa materia)
    checkpoint = pd.concat([checkpoint, new_results_df]).drop_duplicates(subset=['Id'])
    checkpoint.to_csv(ckpt_file, index=False)

    # Cache globale: aggiunge solo i nuovi CELEX con status ok
    new_ok = new_results_df[new_results_df['text_status'] == 'ok'][['Label'] + TEXT_COLS].copy()
    if not new_ok.empty:
        new_to_add   = new_ok[~new_ok['Label'].isin(global_cache['Label'])]
        global_cache = pd.concat([global_cache, new_to_add], ignore_index=True)
        global_cache.to_csv(global_cache_file, index=False)


if len(nodes_todo) == 0:
    print("Tutti i nodi gia' coperti dalla cache — nessun fetch necessario.")
else:
    results = []
    total   = len(nodes_todo)
    n_ok    = 0
    n_err   = 0

    print(f"Inizio fetch: {total} nodi")
    print(f"Tempo stimato: ~{total * DELAY_SECONDS / 60:.0f} minuti\n")

    for i, (_, row) in enumerate(nodes_todo.iterrows()):
        node_id   = row['Id']
        celex     = row.get('Label', row.get('celex_id', ''))
        extracted = extract_all_sections(celex)
        n_ok  += (extracted['text_status'] == 'ok')
        n_err += (extracted['text_status'] != 'ok')
        results.append({'Id': node_id, 'Label': celex, **extracted})

        if (i + 1) % 10 == 0 or (i + 1) == total:
            pct = (i + 1) / total * 100
            print(f"  [{i+1:>4}/{total}] {pct:5.1f}%  ok: {n_ok}  errori: {n_err}")

        if (i + 1) % CHECKPOINT_EVERY == 0:
            batch = pd.DataFrame(results)
            _save_checkpoint_and_cache(batch)
            print(f"  --> ckpt: {len(checkpoint)} nodi | cache: {len(global_cache)} CELEX")

        time.sleep(DELAY_SECONDS)

    _save_checkpoint_and_cache(pd.DataFrame(results))
    print(f"\nFetch completato.")
    print(f"  OK:            {(checkpoint['text_status'] == 'ok').sum()}")
    print(f"  Not found:     {(checkpoint['text_status'] == 'not_found').sum()}")
    print(f"  Cache globale: {len(global_cache)} CELEX totali")

Tutti i nodi gia' coperti dalla cache — nessun fetch necessario.


## 8. Fallback per Atti Non Recuperati

- **Legacy**: atti ante-2000 (template HTML pre-ELI)
- **CaseLaw**: sentenze CGUE
- **Treaty**: trattati fondativi (in genere non disponibili su EUR-Lex)

I fallback aggiornano sia il checkpoint locale che la cache globale.

In [9]:
def _segments_from_flat_text(preamble_text, articles_raw):
    preamble_segs, article_segs = [], []
    if preamble_text:
        parts = re.split(r'\((\d+)\)\s+', preamble_text)
        if len(parts) > 1:
            header = parts[0].strip()
            if header and len(header) >= 30:
                preamble_segs.append({'tipo': 'preambolo_header',
                                      'identificatore': '0', 'testo': header})
            i = 1
            while i < len(parts) - 1:
                num, corpo = parts[i].strip(), parts[i+1].strip()
                if corpo and len(corpo) >= 20:
                    preamble_segs.append({'tipo': 'considerando',
                                          'identificatore': num, 'testo': corpo})
                i += 2
        else:
            preamble_segs.append({'tipo': 'preambolo', 'identificatore': '0',
                                   'testo': preamble_text})
    if articles_raw:
        art_matches = list(re.finditer(
            r'(?:^|\s)(Article\s+(\d+[a-z]?))', articles_raw, re.IGNORECASE))
        if art_matches:
            for j, m in enumerate(art_matches):
                end   = art_matches[j+1].start() if j+1 < len(art_matches) else len(articles_raw)
                testo = articles_raw[m.start():end].strip()
                if len(testo) >= 20:
                    article_segs.append({'tipo': 'articolo',
                                         'identificatore': m.group(2), 'testo': testo})
        else:
            article_segs.append({'tipo': 'articolo', 'identificatore': '?',
                                  'testo': articles_raw})
    return preamble_segs, article_segs, []


def _pack_result(title, preamble_segs, article_segs, annex_segs):
    preamble_text = ' \n '.join(s['testo']   for s in preamble_segs) if preamble_segs else None
    articles_text = ' \n\n '.join(s['testo'] for s in article_segs)  if article_segs  else None
    annexes_text  = ' \n\n '.join(s['testo'] for s in annex_segs)    if annex_segs    else None
    segments = build_segments(preamble_segs, article_segs, annex_segs)
    sections_found = ','.join(filter(None, [
        'title' if title else None, 'preamble' if preamble_segs else None,
        'articles' if article_segs else None, 'annexes' if annex_segs else None,
    ]))
    parti = []
    if title:         parti.append(f"[TITLE] {title}")
    if preamble_text: parti.append(f"[PREAMBLE] {preamble_text}")
    if articles_text: parti.append(f"[ARTICLES] {articles_text}")
    if annexes_text:  parti.append(f"[ANNEXES] {annexes_text}")
    if not parti:
        return None
    full_text = ' \n\n '.join(parti)
    return {
        'title': title, 'preamble': preamble_text,
        'articles': articles_text, 'annexes': annexes_text,
        'full_text': full_text,
        'segments': json.dumps(segments, ensure_ascii=False),
        'n_segments': len(segments),
        'sections_found': sections_found,
        'text_status': 'ok',
        'text_length': len(full_text),
    }


def _load_flat_body(celex, title_classes):
    """Fetch + estrazione corpo testo flat. Usato da extract_legacy e extract_caselaw."""
    try:
        html = eurlex.get_html_by_celex_id(celex, language='en')
        if not html or len(html) < 300:
            return None, None
        soup = BeautifulSoup(html, 'html.parser')
    except Exception:
        return None, None
    title = None
    for css in title_classes:
        tags = soup.find_all('p', class_=css)
        if tags:
            t = ' '.join(t.get_text(separator=' ', strip=True) for t in tags[:3])
            if len(t) > 20:
                title = t[:MAX_TITLE_CHARS]
                break
    if not title:
        for sel in [soup.find('h1'), soup.find('title')]:
            if sel:
                t = re.sub(r'\s*[-\u2013|]\s*EUR-Lex.*$', '',
                           sel.get_text(separator=' ', strip=True), flags=re.IGNORECASE)
                if len(t) > 20:
                    title = t[:MAX_TITLE_CHARS]
                    break
    container = None
    for sel in [{'id': 'TexteOnly'}, {'id': 'document1'}, {'id': 'docHtml'},
                {'class': 'texte'}, {'class': 'doc-content'}]:
        container = soup.find('div', sel)
        if container:
            break
    if not container:
        container = soup.find('body')
    if not container:
        return title, None
    for tag in container.find_all(['nav', 'header', 'footer', 'script', 'style', 'noscript']):
        tag.decompose()
    body = re.sub(r'\s+', ' ', container.get_text(separator=' ', strip=True)).strip()
    return title, body if len(body) >= 100 else None


def extract_legacy(celex):
    title, body = _load_flat_body(
        celex, ['Title', 'Titre', 'sti-tit', 'titredoc', 'doc-ti', 'oj-doc-ti'])
    if body is None:
        return None
    body_upper = body.upper()
    split_pos  = next((body_upper.find(m) for m in PREAMBLE_END_MARKERS
                       if body_upper.find(m) != -1), None)
    preamble_text = body[:split_pos].strip() if split_pos else body[:2000]
    articles_raw  = body[split_pos:].strip() if split_pos else body[2000:]
    preamble_text = preamble_text if len(preamble_text) > 50 else None
    articles_raw  = articles_raw  if len(articles_raw)  > 20 else None
    p, a, x = _segments_from_flat_text(preamble_text, articles_raw)
    return _pack_result(title, p, a, x)


def extract_caselaw(celex):
    title, body = _load_flat_body(
        celex, ['C01Title', 'C01Titre', 'Title', 'oj-doc-ti', 'doc-ti'])
    if body is None:
        return None
    operative_markers = ['ON THOSE GROUNDS', 'THE COURT HEREBY RULES:',
                         'THE COURT (', 'FOR THOSE REASONS,']
    body_upper = body.upper()
    split_pos  = next((body_upper.find(m) for m in operative_markers
                       if body_upper.find(m) != -1), None)
    preamble_text = body[:split_pos].strip() if split_pos else body
    articles_raw  = body[split_pos:].strip() if split_pos else None
    preamble_text = preamble_text if preamble_text and len(preamble_text) > 50 else None
    articles_raw  = articles_raw  if articles_raw  and len(articles_raw)  > 20 else None
    p, a, x = _segments_from_flat_text(preamble_text, articles_raw)
    return _pack_result(title, p, a, x)


print("Estrattori fallback definiti.")

Estrattori fallback definiti.


In [10]:
checkpoint_current = pd.read_csv(ckpt_file, low_memory=False) \
                     if os.path.exists(ckpt_file) else checkpoint.copy()

failed = checkpoint_current[checkpoint_current['text_status'] != 'ok'].copy()
# Escludi CELEX gia' entrati in cache globale nel frattempo
failed = failed[~failed['Label'].astype(str).isin(set(global_cache['Label'].astype(str)))]

print(f"Nodi da ritentare con fallback: {len(failed)}")

if len(failed) > 0:
    if 'LegalType' in nodes.columns:
        fm          = failed.merge(nodes[['Id', 'LegalType']], on='Id', how='left')
        caselaw_ids = set(fm[fm['LegalType'] == 'Case_Law']['Id'])
        treaty_ids  = set(fm[fm['LegalType'] == 'Treaty']['Id'])
        legacy_ids  = set(fm['Id']) - caselaw_ids - treaty_ids
        print(f"  Legacy: {len(legacy_ids)}  CaseLaw: {len(caselaw_ids)}  Treaty: {len(treaty_ids)}")
    else:
        legacy_ids, caselaw_ids, treaty_ids = set(failed['Id']), set(), set()

    print(f"Tempo stimato: ~{len(failed) * DELAY_SECONDS / 60:.0f} minuti")

    n_recovered, n_still_failed = 0, 0
    fallback_batch = []

    for i, (_, row) in enumerate(failed.iterrows()):
        node_id = row['Id']
        celex   = str(row.get('Label', ''))

        result = (extract_caselaw(celex) if node_id in caselaw_ids
                  else None              if node_id in treaty_ids
                  else extract_legacy(celex))

        if result:
            n_recovered += 1
            checkpoint_current.loc[
                checkpoint_current['Id'] == node_id, list(result.keys())
            ] = list(result.values())
            fallback_batch.append({'Id': node_id, 'Label': celex, **result})
        else:
            n_still_failed += 1

        if (i + 1) % 10 == 0 or (i + 1) == len(failed):
            pct = (i + 1) / len(failed) * 100
            print(f"  [{i+1:>4}/{len(failed)}] {pct:5.1f}%  "
                  f"recuperati: {n_recovered}  ancora falliti: {n_still_failed}")

        if (i + 1) % CHECKPOINT_EVERY == 0 and fallback_batch:
            _save_checkpoint_and_cache(pd.DataFrame(fallback_batch))
            fallback_batch = []
            print("  --> Checkpoint+cache aggiornati")

        time.sleep(DELAY_SECONDS)

    if fallback_batch:
        _save_checkpoint_and_cache(pd.DataFrame(fallback_batch))
    checkpoint_current.to_csv(ckpt_file, index=False)

    print(f"\nFallback completato.  Recuperati: {n_recovered}  Mancanti: {n_still_failed}")
    print(f"Cache globale: {len(global_cache)} CELEX totali")

Nodi da ritentare con fallback: 0


## 9. Merge con Metadati e Salvataggio Finale

Priorità nella lookup testi:
1. Cache globale (testi già validati, qualsiasi materia)
2. Checkpoint locale (estratti in questa sessione)
3. Nodi non trovati → `text_status = not_processed`

In [11]:
global_cache_final = pd.read_csv(global_cache_file, low_memory=False)
ckpt_final         = pd.read_csv(ckpt_file, low_memory=False) \
                     if os.path.exists(ckpt_file) else pd.DataFrame()

# Lookup CELEX -> testo: cache globale ha priorita'
text_lookup = (
    global_cache_final
    .drop_duplicates(subset='Label', keep='first')
    .set_index('Label')[TEXT_COLS]
)
if not ckpt_final.empty:
    ckpt_ok = ckpt_final[
        (ckpt_final['text_status'] == 'ok') &
        (~ckpt_final['Label'].isin(text_lookup.index))
    ].set_index('Label')[TEXT_COLS]
    text_lookup = pd.concat([text_lookup, ckpt_ok])

nodes_enriched = nodes.copy().join(text_lookup, on=celex_col, how='left')
nodes_enriched['text_status'] = nodes_enriched['text_status'].fillna('not_processed')
nodes_enriched['n_segments']  = nodes_enriched['n_segments'].fillna(0).astype(int)
nodes_enriched['text_length'] = nodes_enriched['text_length'].fillna(0).astype(int)

nodes_enriched.to_csv(output_file, index=False)

print(f"Salvato: {output_file}")
print(f"Righe:   {len(nodes_enriched)}")
print(f"\nDistribuzione status:")
print(nodes_enriched['text_status'].value_counts().to_string())
print(f"\nCache globale: {len(global_cache_final)} CELEX  ({global_cache_file})")

Salvato: ..\data\output\fdi_screening\nodes_texts.csv
Righe:   19

Distribuzione status:
text_status
ok    19

Cache globale: 1794 CELEX  (..\data\processed\nodes_texts.csv)


## 10. Diagnostica

In [12]:
ok_mask = nodes_enriched['text_status'] == 'ok'

print("=" * 50)
print("SEZIONI TROVATE")
print("=" * 50)
print(nodes_enriched['sections_found'].value_counts().head(10).to_string())

print("\n" + "=" * 50)
print("STATISTICHE SEGMENTI")
print("=" * 50)
print(f"  Totale:      {nodes_enriched.loc[ok_mask, 'n_segments'].sum():,}")
print(f"  Media/atto:  {nodes_enriched.loc[ok_mask, 'n_segments'].mean():.1f}")
print(f"  Max/atto:    {nodes_enriched.loc[ok_mask, 'n_segments'].max():.0f}")

if 'LegalType' in nodes_enriched.columns:
    print("\n" + "=" * 50)
    print("COPERTURA PER TIPO")
    print("=" * 50)
    cov = nodes_enriched.groupby('LegalType').agg(
        totale=('Id', 'count'),
        con_testo=('text_status', lambda x: (x == 'ok').sum()),
        seg_medi=('n_segments', 'mean')
    )
    cov['copertura_%'] = (cov['con_testo'] / cov['totale'] * 100).round(1)
    cov['seg_medi']    = cov['seg_medi'].round(1)
    print(cov.sort_values('totale', ascending=False).to_string())

no_text = nodes_enriched[~ok_mask]
if len(no_text) > 0:
    print(f"\nNodi senza testo: {len(no_text)}")
    if 'LegalType' in no_text.columns:
        print(no_text['LegalType'].value_counts().to_string())

SEZIONI TROVATE
sections_found
title,preamble,articles    19

STATISTICHE SEGMENTI
  Totale:      1,249
  Media/atto:  65.7
  Max/atto:    521

COPERTURA PER TIPO
            totale  con_testo  seg_medi  copertura_%
LegalType                                           
Regulation      16         16      69.8        100.0
Decision         3          3      44.3        100.0


## 11. Verifica Qualità su Campione

In [13]:
from collections import Counter

sample = nodes_enriched[nodes_enriched['text_status'] == 'ok'].sample(3, random_state=42)

for _, row in sample.iterrows():
    celex = row.get('Label', row['Id'])
    segs  = json.loads(row['segments']) if pd.notna(row.get('segments')) else []
    tipi  = Counter(s['tipo'] for s in segs)
    print(f"{'='*60}")
    print(f"CELEX: {celex}")
    print(f"Testo: {row['text_length']:,} car. | Segmenti: {row['n_segments']} {dict(tipi)}")
    print(f"Titolo: {str(row.get('title', ''))[:120]}")
    shown = set()
    for s in segs:
        if s['tipo'] not in shown:
            shown.add(s['tipo'])
            print(f"  [{s['tipo']} {s['identificatore']}]: {s['testo'][:200]}")
        if len(shown) >= 4:
            break
    print()

CELEX: 32010R1227
Testo: 1,920 car. | Segmenti: 9 {'preambolo_header': 1, 'considerando': 6, 'articolo': 2}
Titolo: COMMISSION REGULATION (EU) No 1227/2010 of 20 December 2010 amending Regulation (EC) No 1055/2008 implementing Regulatio
  [preambolo_header 0]: THE EUROPEAN COMMISSION, Having regard to the Treaty on the Functioning of the European Union, Having regard to Regulation (EC) No 184/2005 of the European Parliament and of the Council of 12 January 
  [considerando 1]: Regulation (EC) No 184/2005 establishes a common framework for the systematic production of Community statistics concerning balance of payments, international trade in services and foreign direct inve
  [articolo 1]: Article 1 Regulation (EC) No 1055/2008 is amended as follows: 1. Article 2 is replaced by the following: ‘Member States shall supply their quality report not later than 31 May every year.’; 2. the Ann

CELEX: 32023R1441
Testo: 56,304 car. | Segmenti: 175 {'preambolo_header': 1, 'considerando': 27, 'a